# PIR End-to-End Recommender Pipeline

This notebook implements the Kaggle-standard Two-Stage Recommender Pipeline (Retrieval + Ranking) to solve the Personalized Item Recommendation (PIR) problem. 
It integrates the insights derived from the 50 analytical hypotheses to build a high-precision LightGBM/XGBoost ranker.


In [ ]:
import polars as pl
import numpy as np
import lightgbm as lgb
import xgboost as xgb
from scipy.sparse import csr_matrix
from sklearn.decomposition import TruncatedSVD
from typing import List, Set, Dict, Tuple
import warnings
import gc
import time
warnings.filterwarnings('ignore')

# Set display options
pl.Config.set_tbl_rows(10)


## 1. Data Loading and Time Splits
We load the transaction data and split it based on the strict Out-of-Time validation protocol:
- **Train (History)**: Jan - Oct 2025
- **Validation (Train Ranker)**: Nov 2025
- **Test (Evaluate Ranker)**: Dec 2025


In [ ]:
def load_and_prep_data(transaction_path):
    # Fast load using Polars
    df = pl.scan_parquet(transaction_path).select([
        pl.col('customer_id').cast(pl.Int64),
        pl.col('item_id').cast(pl.Utf8),
        pl.col('event_date').cast(pl.Datetime).alias('event_ts'),
        pl.col('quantity').cast(pl.Float64).fill_null(0.0),
        pl.col('event_type').cast(pl.Utf8).str.to_lowercase()
    ]).filter(
        (pl.col('event_type') == 'purchased') | (pl.col('quantity') > 0)
    ).with_columns(
        pl.col('event_ts').dt.month().alias('month')
    ).collect()
    return df

df_raw = load_and_prep_data('../transaction_full_2025.parquet')
items_df = pl.read_parquet('../items.parquet')

print(f"Total transactions: {df_raw.height:,}")


In [ ]:
df_train_base = df_raw.filter(pl.col('month') <= 10)
df_val_truth  = df_raw.filter(pl.col('month') == 11)

df_train_val_base = df_raw.filter(pl.col('month') <= 11)
df_test_truth = df_raw.filter(pl.col('month') == 12)

print(f"Train base (Jan-Oct): {df_train_base.height:,}")
print(f"Val truth (Nov): {df_val_truth.height:,}")
print(f"Test truth (Dec): {df_test_truth.height:,}")


## 2. Stage 1: Candidate Generation (Retrieval)
We retrieve ~200 candidates per user to maximize the chances of capturing the items they actually bought (Total Correct Hits). We use:
1. **Matrix Factorization (SVD)**: Good for collaborative filtering.
2. **Replenishment**: Items the user bought before (Idea 6).
3. **Popularity Baseline**: Global bestsellers to handle cold-start.


In [ ]:
class SVDRetriever:
    def __init__(self, n_components=64):
        self.model = TruncatedSVD(n_components=n_components, random_state=42)
        
    def fit(self, history_df):
        # Build user-item interaction weights
        interactions = history_df.group_by(['customer_id', 'item_id']).agg(pl.col('quantity').sum().alias('weight'))
        
        self.users = interactions['customer_id'].unique().to_list()
        self.items = interactions['item_id'].unique().to_list()
        
        self.user2idx = {u: i for i, u in enumerate(self.users)}
        self.item2idx = {i: idx for idx, i in enumerate(self.items)}
        
        row_idx = [self.user2idx[u] for u in interactions['customer_id']]
        col_idx = [self.item2idx[i] for i in interactions['item_id']]
        
        self.matrix = csr_matrix((interactions['weight'].to_numpy(), (row_idx, col_idx)), shape=(len(self.users), len(self.items)))
        
        self.user_factors = self.model.fit_transform(self.matrix)
        self.item_factors = self.model.components_.T
        
        # Global popularity for fallback
        self.pop_items = interactions.group_by('item_id').agg(pl.col('weight').sum()).sort('weight', descending=True)['item_id'].head(100).to_list()
        return self
    
    def retrieve(self, target_users, top_k=100):
        candidates = []
        for u in target_users:
            if u in self.user2idx:
                u_idx = self.user2idx[u]
                scores = self.user_factors[u_idx] @ self.item_factors.T
                top_indices = np.argsort(-scores)[:top_k]
                retrieved = [self.items[i] for i in top_indices]
            else:
                retrieved = self.pop_items[:top_k]
            
            for i in retrieved:
                candidates.append({'customer_id': u, 'item_id': i, 'svd_rank': 1})
        
        return pl.DataFrame(candidates)


In [ ]:
def generate_candidates(history_df, target_users, top_k=100):
    print("Fitting SVD Retriever...")
    svd = SVDRetriever(n_components=64).fit(history_df)
    
    print("Generating SVD Candidates...")
    df_cands_svd = svd.retrieve(target_users, top_k=top_k)
    
    print("Generating Replenishment Candidates...")
    df_rep = history_df.filter(pl.col('customer_id').is_in(target_users)).select(['customer_id', 'item_id']).unique()
    df_rep = df_rep.with_columns(pl.lit(1).alias('is_rep'))
    
    print("Generating Global Popularity Candidates...")
    pop_items = svd.pop_items[:50]
    pop_cands = pl.DataFrame([{'customer_id': u} for u in target_users]).join(pl.DataFrame({'item_id': pop_items}), how='cross')
    pop_cands = pop_cands.with_columns(pl.lit(1).alias('is_pop'))
    
    print("Combining Candidates...")
    all_cands = pl.concat([
        df_cands_svd.select(['customer_id', 'item_id']),
        df_rep.select(['customer_id', 'item_id']),
        pop_cands.select(['customer_id', 'item_id'])
    ]).unique(subset=['customer_id', 'item_id'])
    
    return all_cands


## 3. Stage 2: Feature Engineering (The Feature Store)
Here we compute robust features from the analytical roadmap to empower the Ranker.


In [ ]:
def build_features(history_df, candidates_df, items_df):
    print("Extracting User Features...")
    user_feats = history_df.group_by('customer_id').agg([
        pl.col('item_id').n_unique().alias('user_unique_items'),
        pl.len().alias('user_total_txns'),
        (pl.col('item_id').n_unique() / pl.len()).alias('user_exploration_ratio') # Proxies 'Explorer' vs 'Loyalist'
    ])
    
    print("Extracting Item Features...")
    item_feats = history_df.group_by('item_id').agg([
        pl.len().alias('item_global_pop'),
        pl.col('customer_id').n_unique().alias('item_unique_buyers')
    ])
    
    print("Extracting Interaction Features...")
    inter_feats = history_df.group_by(['customer_id', 'item_id']).agg([
        pl.len().alias('ui_buy_count'),
        (pl.col('event_ts').max() - pl.col('event_ts').min()).dt.total_days().alias('ui_buy_duration_days')
    ])
    
    print("Joining Features to Candidates...")
    df_feat = candidates_df.join(user_feats, on='customer_id', how='left')
    df_feat = df_feat.join(item_feats, on='item_id', how='left')
    df_feat = df_feat.join(inter_feats, on=['customer_id', 'item_id'], how='left')
    
    # Fill Nulls for items/interactions not seen before
    df_feat = df_feat.fill_null(0)
    
    return df_feat


## 4. Constructing Train and Test Datasets


In [ ]:
def create_dataset(history_df, truth_df, items_df, sample_users=None):
    target_users = truth_df['customer_id'].unique().to_list()
    if sample_users:
        np.random.seed(42)
        target_users = list(np.random.choice(target_users, sample_users, replace=False))
        truth_df = truth_df.filter(pl.col('customer_id').is_in(target_users))
    
    candidates = generate_candidates(history_df, target_users, top_k=100)
    print(f"Generated {candidates.height:,} candidate pairs for {len(target_users):,} users.")
    
    dataset = build_features(history_df, candidates, items_df)
    
    # Labels
    truth_pairs = truth_df.select(['customer_id', 'item_id']).unique().with_columns(pl.lit(1).alias('target'))
    dataset = dataset.join(truth_pairs, on=['customer_id', 'item_id'], how='left').fill_null(0)
    
    print(f"Dataset created. Positive Ratio: {dataset['target'].mean():.4f}\n")
    return dataset

print("=== BUILDING TRAINING DATA (Eval on Nov) ===")
# Using a sample for quick training in this notebook
train_data = create_dataset(df_train_base, df_val_truth, items_df, sample_users=10000)

print("=== BUILDING TEST DATA (Eval on Dec) ===")
test_data = create_dataset(df_train_val_base, df_test_truth, items_df, sample_users=10000)


## 5. Stage 3: Train Ranking Models (LightGBM vs XGBoost)


In [ ]:
feature_cols = [
    'user_unique_items', 'user_total_txns', 'user_exploration_ratio',
    'item_global_pop', 'item_unique_buyers', 
    'ui_buy_count', 'ui_buy_duration_days'
]

X_train = train_data.select(feature_cols).to_pandas()
y_train = train_data['target'].to_pandas()

X_test = test_data.select(feature_cols).to_pandas()
y_test = test_data['target'].to_pandas()

print("Training LightGBM...")
lgb_model = lgb.LGBMClassifier(n_estimators=300, learning_rate=0.05, random_state=42, n_jobs=-1)
lgb_model.fit(X_train, y_train, eval_set=[(X_test, y_test)], eval_metric='logloss')

print("\nTraining XGBoost...")
xgb_model = xgb.XGBClassifier(n_estimators=300, learning_rate=0.05, random_state=42, n_jobs=-1)
xgb_model.fit(X_train, y_train, eval_set=[(X_test, y_test)], verbose=False)

print("\nModels trained successfully.")


## 6. Stage 4: Inference and Final Evaluation (Dec 2025)
We compute Precision@10, MRR, IoU, and Total Hits on the Top 10 predictions per user.


In [ ]:
test_data = test_data.with_columns([
    pl.Series(name='pred_lgb', values=lgb_model.predict_proba(X_test)[:, 1]),
    pl.Series(name='pred_xgb', values=xgb_model.predict_proba(X_test)[:, 1])
])

def evaluate_model(model_col):
    # Get Top 10
    top10 = (
        test_data
        .sort(['customer_id', model_col], descending=[False, True])
        .group_by('customer_id', maintain_order=True)
        .head(10)
    )
    
    # Truth
    truth_map = df_test_truth.filter(pl.col('customer_id').is_in(top10['customer_id'].unique().to_list()))
    truth_map = truth_map.group_by('customer_id').agg(pl.col('item_id'))
    truth_dict = {row[0]: set(row[1]) for row in truth_map.iter_rows()}
    
    # Preds
    pred_map = top10.group_by('customer_id').agg(pl.col('item_id'))
    pred_dict = {row[0]: list(row[1]) for row in pred_map.iter_rows()}
    
    total_hits, mrr_sum, p10_sum, iou_sum, map_sum = 0, 0.0, 0.0, 0.0, 0.0
    n_users = len(truth_dict)
    
    if n_users == 0: return {}
    
    for uid, truth in truth_dict.items():
        preds = pred_dict.get(uid, [])
        hits = [p for p in preds if p in truth]
        total_hits += len(hits)
        
        p10 = len(hits) / 10.0 if len(preds) > 0 else 0.0
        p10_sum += p10
        
        mrr = 0.0
        for i, p in enumerate(preds):
            if p in truth:
                mrr = 1.0 / (i + 1)
                break
        mrr_sum += mrr
        
        intersection = len(set(preds) & truth)
        union = len(set(preds) | truth)
        iou_sum += intersection / union if union > 0 else 0.0
        
        # MAP@10
        ap_hits, ap_sum = 0, 0.0
        for i, p in enumerate(preds):
            if p in truth:
                ap_hits += 1
                ap_sum += ap_hits / (i + 1)
        map_sum += ap_sum / min(len(truth), 10) if len(truth) > 0 else 0.0
        
    return {
        'Total Correct Hits': total_hits,
        'Precision@10': p10_sum / n_users,
        'MAP': map_sum / n_users,
        'MRR': mrr_sum / n_users,
        'IoU': iou_sum / n_users
    }

print("=== LightGBM Results ===")
lgb_res = evaluate_model('pred_lgb')
for k, v in lgb_res.items(): print(f"{k}: {v:.4f}" if isinstance(v, float) else f"{k}: {v}")

print("\n=== XGBoost Results ===")
xgb_res = evaluate_model('pred_xgb')
for k, v in xgb_res.items(): print(f"{k}: {v:.4f}" if isinstance(v, float) else f"{k}: {v}")
